In [ ]:
import pandas as pd

# load the dataset divided by country(Iran, Israel, Egypt, Saudi Arabia)
# keep_default_na=False keep the empty value and None value not to be NaN
df_all = pd.read_csv('./data_mideast/MidEast.csv', keep_default_na=False)

/tmp/ipykernel_945721/2010097251.py:6: DtypeWarning: Columns (4,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv('./data_mideast_new/MidEast.csv', keep_default_na=False)


In [2]:
len(df_all)

22435752

In [3]:
# Remove Rows Lacking Entity Information
original_rows = df_all.shape[0]
print(f"Original rows: {original_rows}")
# Actor/Recipient Name is Nan
df = df_all[((df_all['Actor Name']!="") & (df_all['Recipient Name']!=""))]
#df = df_all[((df_all['Actor Name'].notnull()) & (df_all['Recipient Name'].notnull())) | ((df_all['Actor Name Raw'].notnull()) & (df_all['Recipient Name Raw'].notnull()))]
name_nan = original_rows-df.shape[0]
print(f"Name/Name Raw == Nan: {name_nan}")
new_rows = df.shape[0]
print(f"New rows: {new_rows}")

# Actor Name/Name Raw Statistic
actor_none_notnone = (df['Actor Name']=="None") & (df['Actor Name Raw']!="None")
actor_none_none = (df['Actor Name']=="None") & (df['Actor Name Raw'] == "None")
actor_none_notnone_sum = actor_none_notnone.sum()
actor_none_none_sum = actor_none_none.sum()

print(f"[Actor Name] is 'None' but [Actor Name Raw] is not 'None': {actor_none_notnone_sum} ({actor_none_notnone_sum/new_rows*100:.4f})%")
print(f"[Actor Name] is 'None and [Actor Name Raw] is 'None': {actor_none_none_sum} ({actor_none_none_sum/new_rows*100:.4f})%")

# Recipient Name/Name Raw Statistic
recipient_none_notnone = (df['Recipient Name']=="None") & (df['Recipient Name Raw']!="None")
recipient_none_none = (df['Recipient Name']=="None") & (df['Recipient Name Raw'] == "None")
recipient_none_notnone_sum = recipient_none_notnone.sum()
recipient_none_none_sum = recipient_none_none.sum()

print(f"[Recipient Name] is 'None' but [Recipient Name Raw] is not 'None': {recipient_none_notnone_sum} ({recipient_none_notnone_sum/new_rows*100:.4f})%")
print(f"[Recipient Name] is 'None and [Recipient Name Raw] is 'None': {recipient_none_none_sum} ({recipient_none_none_sum/new_rows*100:.4f})%")

to_be_dropped = (recipient_none_none) | (actor_none_none)
to_be_dropped_sum = to_be_dropped.sum()
print(f"Rows to be dropped: {to_be_dropped_sum} ({to_be_dropped_sum/original_rows*100:.4f})%")
df = df[~to_be_dropped]

Original rows: 22435752
Name/Name Raw == Nan: 851259
New rows: 21584493
[Actor Name] is 'None' but [Actor Name Raw] is not 'None': 7372471 (34.1563)%
[Actor Name] is 'None and [Actor Name Raw] is 'None': 1701996 (7.8853)%
[Recipient Name] is 'None' but [Recipient Name Raw] is not 'None': 6688892 (30.9893)%
[Recipient Name] is 'None and [Recipient Name Raw] is 'None': 5682720 (26.3278)%
Rows to be dropped: 7384716 (32.9149)%


In [4]:
from tqdm import tqdm

def split_merge_entities(row, field, raw_field):
    entities = row[field].split(';')
    isReplace = False
    def format_ents(ents):
        ents_format = []
        for ent in ents:
            ent = ent.strip().strip('"').strip("'")
            if ent != "None":
                ents_format.append(ent)
        return ents_format
    
    entities_format = format_ents(entities)

    if len(entities_format) == 0:
        isReplace = True
        entities_raw = row[raw_field].split(';')
        entities_format = format_ents(entities_raw)

    return entities_format,isReplace

def expand_rows(df, field, raw_field):
    # Create a new DataFrame where the 'field' values are split and expanded into separate rows
    expanded_rows = []
    df[field+' isReplace'] = False
    for index, row in tqdm(df.iterrows(), total=len(df), desc=f"Expanding {field}"):
        entities,isReplace = split_merge_entities(row, field, raw_field)
        for entity in entities:
            new_row = row.copy()
            new_row[field] = entity
            new_row[field+' isReplace'] = isReplace
            expanded_rows.append(new_row)
    return pd.DataFrame(expanded_rows)

print(f"Original rows: {df.shape[0]}")

# Expand rows for 'Actor Name' and 'Recipient Name' with their respective raw fields
df_expand_actor = expand_rows(df, 'Actor Name', 'Actor Name Raw')
df_expand = expand_rows(df_expand_actor, 'Recipient Name', 'Recipient Name Raw')

# Print the row counts before and after processing
print(f"Resulting rows: {len(df_expand)}")

# Save the processed dataset back to a CSV file
df_expand.to_csv('./data_mideast/MidEast_expanded.csv', index=False)

Original rows: 14199777


Expanding Recipient Name: 100%|██████████| 16234370/16234370 [28:50<00:00, 9381.82it/s] 


Resulting rows: 19744507


In [5]:
import pandas as pd

# deduplicate
# Load the expanded dataset from CSV
df_expand = pd.read_csv('./data_mideast/MidEast_expanded.csv')

# Display original number of rows
original_rows = len(df_expand)
print(f"Original rows before removing duplicates: {original_rows}")

# Remove duplicates based on all specified columns being the same
df_deduplicated = df_expand.drop_duplicates(subset=['Actor Name', 'Recipient Name', 'Event Type','Event Mode', 'Event Date'])
# df = df.drop_duplicates(subset=['Actor Name', 'Recipient Name', 'Event Type', 'Event Date'])

# Display number of rows after removing duplicates
resulting_rows = len(df_deduplicated)
print(f"Resulting rows after removing duplicates: {resulting_rows}")

# Save the deduplicated dataset back to a CSV file
df_deduplicated.to_csv('./data_mideast/MidEast_deduplicated.csv', index=False)

/tmp/ipykernel_945721/64325979.py:5: DtypeWarning: Columns (37,38,39,40,41,42,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_expand = pd.read_csv('./data_mideast_new/MidEast_expanded.csv')


Original rows before removing duplicates: 19744507
Resulting rows after removing duplicates: 9740791


In [7]:
import pandas as pd

df_deduplicated = pd.read_csv('./data_mideast/MidEast_deduplicated.csv')
entitis_actor = set(df_deduplicated[(df_deduplicated['Actor Name isReplace'] == True)]['Actor Name'].unique())
entitis_recipient = set(df_deduplicated[(df_deduplicated['Recipient Name isReplace'] == True)]['Recipient Name'].unique())
unique_entities = set(entitis_actor).union(entitis_recipient)

/tmp/ipykernel_136470/4097816524.py:3: DtypeWarning: Columns (37,38,39,40,41,42,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_deduplicated = pd.read_csv('./data_mideast/MidEast_deduplicated.csv')


In [8]:
len(unique_entities)

632015

In [ ]:
from openai import OpenAI


# client = OpenAI()

client = OpenAI(api_key="YOUR_OPENAI_KEY")

def check_entity(entity):
  response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "Your task is to determine whether the input content is suitable as a subject or object in an event. Output \"YES\" if the input is a person, country, organization, institution, etc., and \"NO\" if the input is a number, date, verb, sentence, etc. Here are some examples:\n\nInput: Israel  \nOutput: YES \n\nInput: Foreign minister  \nOutput: YES \n\nInput: Province casualty  \nOutput: NO \n\nInput: 1988  \nOutput: NO \n\nInput: held  \nOutput: NO \n\nInput: 37th day  \nOutput: NO \n\nInput: Thursday  \nOutput: NO"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text":  "Input: "+entity+"\nOutput: "
        }
      ]
    }
  ],
  temperature=0,
  max_tokens=10,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0
  )
    # print(response.choices[0].message.content)
  result = response.choices[0].message.content
  return True if "yes" in result.lower() else False
  # return result

In [10]:
print("Your task is to determine whether the input content is suitable as a subject or object in an event. Output \"YES\" if the input is a person, country, organization, institution, etc., and \"NO\" if the input is a number, date, verb, sentence, etc. Here are some examples:\n\nInput: Israel  \nOutput: YES \n\nInput: Foreign minister  \nOutput: YES \n\nInput: Province casualty  \nOutput: NO \n\nInput: 1988  \nOutput: NO \n\nInput: held  \nOutput: NO \n\nInput: 37th day  \nOutput: NO \n\nInput: Thursday  \nOutput: NO")

Your task is to determine whether the input content is suitable as a subject or object in an event. Output "YES" if the input is a person, country, organization, institution, etc., and "NO" if the input is a number, date, verb, sentence, etc. Here are some examples:

Input: Israel  
Output: YES 

Input: Foreign minister  
Output: YES 

Input: Province casualty  
Output: NO 

Input: 1988  
Output: NO 

Input: held  
Output: NO 

Input: 37th day  
Output: NO 

Input: Thursday  
Output: NO


In [ ]:
import concurrent.futures
from tqdm import tqdm
import json

def check_entities(entities,max_workers=100):
    results = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(check_entity, entity): entity for entity in entities}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            entity = futures[future]
            try:
                results[entity] = future.result()
            except Exception as exc:
                results[entity] = str(exc)
    return results

# with open('ent_filter.json','r') as f:
#     entit_check_results = json.load(f)
# print(len(entit_check_results))
# check all unique entities
entit_recheck_results = check_entities(unique_entities)
entit_recheck = [ent for ent,v in entit_recheck_results.items() if v==False]
with open('./data_mideast/ent_check/ent_recheck_results.json','w') as f:
    json.dump(entit_recheck_results,f,indent=2,ensure_ascii=False)
with open('./data_mideast_new/ent_check/ent_recheck.json','w') as f:
    json.dump(entit_recheck,f,indent=2,ensure_ascii=False)

 27%|██▋       | 173338/632015 [2:29:03<3:53:15, 32.77it/s] 

In [13]:
len(entit_recheck)

0

In [1]:
import json
import pandas as pd

with open('./data_mideast/ent_check/ent_recheck.json','r') as f:
    ent_recheck = json.load(f)

df_deduplicated = pd.read_csv('./data_mideast/MidEast_deduplicated.csv')

print(len(ent_recheck))

# df_droped = df_deduplicated[((df_deduplicated['Actor Name'].isin(ent_recheck))|(df_deduplicated['Recipient Name'].isin(ent_recheck)))]
df_droped = df_deduplicated[df_deduplicated['Actor Name'].isin(ent_recheck)]
# df_droped['Actor Name'].value_counts()
actor_filter = df_droped[['Actor Name','Event ID']].value_counts().reset_index()
actor_filter.columns = ['ent', 'id','Count']
actor_filter = actor_filter.drop('Count',axis=1)
# actor_filter
actor_event_filter = actor_filter['ent'].value_counts().reset_index()
actor_event_filter.columns = ['ent','Count']
# actor_event_filter
ent_maybe_list_actor = list(actor_event_filter[actor_event_filter['Count']>=5]['ent'].unique())

df_droped = df_deduplicated[df_deduplicated['Recipient Name'].isin(ent_recheck)]
# df_droped['Actor Name'].value_counts()
recipient_filter = df_droped[['Recipient Name','Event ID']].value_counts().reset_index()
recipient_filter.columns = ['ent', 'id','Count']
recipient_filter = recipient_filter.drop('Count',axis=1)
# actor_filter
recipient_event_filter = recipient_filter['ent'].value_counts().reset_index()
recipient_event_filter.columns = ['ent','Count']
# recipient_event_filter
ent_maybe_list_recipient = list(recipient_event_filter[recipient_event_filter['Count']>=5]['ent'].unique())

# ent_maybe_list = [ent for ent in ent_maybe_list_actor if ent in ent_maybe_list_recipient]
ent_maybe_list = list(set(ent_maybe_list_actor + ent_maybe_list_recipient))
with open('./data_mideast/ent_check/ent_maybe_list.json','w') as f:
    json.dump(ent_maybe_list,f,indent=2,ensure_ascii=False)

/tmp/ipykernel_954322/1720008508.py:7: DtypeWarning: Columns (37,38,39,40,41,42,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_deduplicated = pd.read_csv('./data_mideast_new/MidEast_deduplicated.csv')


101170


In [2]:
print(len(ent_maybe_list))

12942


In [ ]:
from openai import OpenAI


# client = OpenAI()

client = OpenAI(api_key="YOUR_API_KEY")

def check_entity(entity):
  response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "Your task is to determine whether the input content is suitable as a subject or object in an event. Output \"YES\" if the input is a person, country, organization, institution, etc., and \"NO\" if the input is a number, date, verb, sentence, etc. Here are some examples:\n\nInput: Israel  \nOutput: YES \n\nInput: Foreign minister  \nOutput: YES \n\nInput: Province casualty  \nOutput: NO \n\nInput: 1988  \nOutput: NO \n\nInput: held  \nOutput: NO \n\nInput: 37th day  \nOutput: NO \n\nInput: Thursday  \nOutput: NO"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text":  "Input: "+entity+"\nOutput: "
        }
      ]
    }
  ],
  temperature=0,
  max_tokens=10,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0
  )
    # print(response.choices[0].message.content)
  result = response.choices[0].message.content
  return True if "yes" in result.lower() else False
  # return result

In [ ]:
import concurrent.futures
from tqdm import tqdm
import json

def check_entities(entities,max_workers=100):
    results = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(check_entity, entity): entity for entity in entities}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            entity = futures[future]
            try:
                results[entity] = future.result()
            except Exception as exc:
                results[entity] = str(exc)
    return results

print(len(ent_maybe_list))
# check all unique entities
entit_final_results = check_entities(ent_maybe_list)
entit_final = [ent for ent,v in entit_final_results.items() if v==False]
with open('./data_mideast/ent_check/ent_final_results.json','w') as f:
    json.dump(entit_final_results,f,indent=2,ensure_ascii=False)
with open('./data_mideast/ent_check/ent_final.json','w') as f:
    json.dump(entit_final,f,indent=2,ensure_ascii=False)

3753


100%|██████████| 3753/3753 [03:01<00:00, 20.63it/s]


In [3]:
import pandas as pd
import json

df_deduplicated = pd.read_csv('./data_mideast/MidEast_deduplicated.csv')

with open('./data_mideast/ent_check/ent_recheck.json','r') as f:
    ent_recheck = json.load(f)
with open('./data_mideast/ent_check/ent_final.json','r') as f:
    ent_final = json.load(f)
with open('./data_mideast/ent_check/ent_final_results.json','r') as f:
    ent_final_results = json.load(f)
ent_final_list = [ent for ent in ent_final_results.keys() if ent not in ent_final] # final YES
ent_recheck_list = [ent for ent in ent_recheck if ent not in ent_final_list]
df_filtered = df_deduplicated[~((df_deduplicated['Actor Name'].isin(ent_recheck_list))|(df_deduplicated['Recipient Name'].isin(ent_recheck_list)))]

df_filtered.shape

/tmp/ipykernel_954322/205057170.py:4: DtypeWarning: Columns (37,38,39,40,41,42,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_deduplicated = pd.read_csv('./data_mideast_new/MidEast_deduplicated.csv')


(8984973, 48)

In [4]:
with open('./data_mideast/ent_check/ent_final_results.json','r') as f:
    ent_final_results = json.load(f)
len(ent_final_results)

12942

In [5]:
df_deduplicated.shape

(9740791, 48)

In [6]:
df_filtered.to_csv('./data_mideast/final/MidEast_filtered.csv', index=False)

In [7]:
print(len(set(df_deduplicated['Actor Name'].unique()).union(set(df_deduplicated['Recipient Name'].unique()))))
print(len(set(df_filtered['Actor Name'].unique()).union(set(df_filtered['Recipient Name'].unique()))))

805951
699967


In [8]:
print(len(ent_recheck_list))
df_droped = df_deduplicated[((df_deduplicated['Actor Name'].isin(ent_recheck_list))|(df_deduplicated['Recipient Name'].isin(ent_recheck_list)))]
drop = set(df_droped['Actor Name'].unique()).union(set(df_droped['Recipient Name'].unique()))
drop_not_in_ent_recheck_list = drop - set(ent_recheck_list)
len(drop_not_in_ent_recheck_list -set(df_filtered['Actor Name'].unique()).union(set(df_filtered['Recipient Name'].unique())))

100548


5436

In [9]:
import pandas as pd

# Load the deduplicated dataset from CSV
df_deduplicated = pd.read_csv('./data_mideast/final/MidEast_filtered.csv')

# Convert the 'Event Date' column to datetime format
df_deduplicated['Event Date'] = pd.to_datetime(df_deduplicated['Event Date'], format='%Y-%m-%d')

# split_date = pd.Timestamp('2024-04-21')
# all_df = df_deduplicated[df_deduplicated['Event Date'] <= split_date]
all_df = df_deduplicated

# Display the number of rows in each set
print(f"Number of rows in  dataset: {len(all_df)}")

# Save the train and test sets to separate CSV files
all_df.to_csv('./data_mideast/final/MidEast_date.csv', index=False)

/tmp/ipykernel_954322/889906493.py:4: DtypeWarning: Columns (37,38,39,40,41,42,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_deduplicated = pd.read_csv('./data_mideast_new/final/MidEast_filtered.csv')


Number of rows in  dataset: 8984973


In [ ]:
import pandas as pd

# Load the all_df dataset from CSV
all_df = pd.read_csv('./data_mideast/final/MidEast_date.csv')

def truncate_string(value):
    return value.split('_')[0]

all_df = all_df.loc[:, ["Event ID", 'Actor Name', 'Event Type', 'Event Mode', 'Recipient Name', "Event Date", "Contexts", "Actor Country", "Recipient Country", "Country", "Story People", "Story Organizations", "Story Locations"]]
all_df["Md5"] = all_df["Event ID"].apply(truncate_string)
all_df = all_df.sort_values(by=['Event Date'], ignore_index=True)
all_df.to_csv('../2_historical_event/MidEast.csv', index=False)

/tmp/ipykernel_954322/1878960144.py:4: DtypeWarning: Columns (37,38,39,40,41,42,45) have mixed types. Specify dtype option on import or set low_memory=False.
  all_df = pd.read_csv('./data_mideast_new/final/MidEast_date.csv')
